In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output

#### methods to load data

In [71]:
# Load data methods
def load_one_trial(trial_file):
    """
    Loads a single trial CSV file into a pandas DataFrame.
    
    Parameters:
    trial_file (str): Path to the target CSV file.
    
    Returns:
    DataFrame: The loaded trial data.
    """
    return pd.read_csv(trial_file)

def load_all_trials(participant_dir):
    """
    Loads and combines all individual trial CSV files for a single participant.
    
    Parameters:
    participant_dir (str): Path to the participant's folder containing CSVs.
    
    Returns:
    DataFrame: A single, continuous DataFrame containing all trials.
    """
    all_dfs = []

    # Loop through all files in the folder
    for file_name in os.listdir(participant_dir):
        # Only process files that end with .csv
        if file_name.endswith('.csv'):
            # Construct the full path to the file
            file_path = os.path.join(participant_dir, file_name)

            # Load the individual trial and add it to our list
            trial_df = load_one_trial(file_path)
            all_dfs.append(trial_df)

    # Combine all individual DataFrames into a single continuous DataFrame
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()  # Return empty DataFrame if no CSV files found
def load_all_participants(data_dir,append=True):
    """
    Iterates through all participant directories within a root data directory.
    
    Parameters:
    data_dir (str): Root directory containing all participant folders.
    append (bool): If True, merges everything into one DataFrame. 
                   If False, returns a list of separate DataFrames.
                   
    Returns:
    DataFrame or list: Combined dataset or a list of participant DataFrames.
    """
    all_dfs = []
    # Loop through all files in the folder
    for participant_dir in os.listdir(data_dir):
            # Construct the full path to the file
            file_path = os.path.join(data_dir, participant_dir)
            participant_df = load_all_trials(file_path)
            participant_df["participant_id"] = participant_dir
            all_dfs.append(participant_df)
    if append:
        return pd.concat(all_dfs,ignore_index=True)
    else:
        return all_dfs



#### Load the data

In [72]:
data_dir = "C:/Users/HP/Documents/uni/SoSe26/expra/code/analysis/data"
data = load_all_participants(data_dir)
data
# correct corrupted data (only do this for ge03 testdata with the bug not fixed)
cursor_x_cm = data["gaze_x"]*(59.7/33) 
cursor_y_cm = data["gaze_y"]
gaze_x = data["cursor_x_cm"]
gaze_y = data["cursor_y_cm"]
data["cursor_x_cm"] = cursor_x_cm
data["cursor_y_cm"] = cursor_y_cm
data["gaze_x"] = gaze_x
data["gaze_y"] = gaze_y
data["block"] = 0
data

,trial,time,cursor_x_pix,cursor_y_pix,gaze_x,gaze_y,cursor_x_cm,cursor_y_cm,target_x,target_y,state_marker,participant_id,block
0,0,3.654690,222.0,-44.0,-70.000000,15.400024,5.177109,-1.029722,434.959442,53.292008,0,ge03,0
1,0,3.686773,222.0,-44.0,-80.699951,35.400024,5.177109,-1.029722,434.959442,53.292008,0,ge03,0
2,0,3.693141,222.0,-44.0,-83.800049,36.700012,5.177109,-1.029722,434.959442,53.292008,0,ge03,0
3,0,3.699954,222.0,-44.0,-88.400024,36.000000,5.177109,-1.029722,434.959442,53.292008,0,ge03,0
4,0,3.706808,222.0,-44.0,-87.699951,37.599976,5.177109,-1.029722,434.959442,53.292008,0,ge03,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7170,7,76.555516,213.0,203.0,199.199951,219.000000,4.967227,4.750764,205.322805,203.775262,4,ge03,0
7171,7,76.562504,213.0,203.0,200.000000,216.899994,4.967227,4.750764,205.322805,203.775262,4,ge03,0
7172,7,76.569477,213.0,203.0,199.300049,216.200012,4.967227,4.750764,205.322805,203.775262,4,ge03,0
7173,7,76.576412,213.0,203.0,197.800049,216.600006,4.967227,4.750764,205.322805,203.775262,4,ge03,0


### methods to visualize trials

In [73]:
def visualize_trial(data,participant_id="ge03",block_n=0,trial_n=0):
    # 1. Load the trial data
    df = data[data["participant_id"] == participant_id]
    #df = data[data["block"] == block_n]
    df = df[df["trial"] == trial_n]


    # 2. Extract unique trial number and target positions for the plot title/markers
    # (Assuming target coordinates and trial ID remain constant within a single trial file)
    target_x = df["target_x"].iloc[0]
    target_y = df["target_y"].iloc[0]

    # 3. Create the plot
    plt.figure(figsize=(10, 8))

    # wrong screen width and pixel amount was used in params leading to conversion factor
    # and cursor_cm is stored in gaze (was switched in the update function by mistake)

    # Plot Cursor Trajectory in cm
    plt.plot(
        df["cursor_x_pix"],
        df["cursor_y_pix"],
        label="Cursor Path",
        color="royalblue",
        linewidth=2,
        marker="o",
        markersize=3,
        alpha=0.7,
    )


    # gaze is actually stored in the cursor variable (was switched in the update function by mistake)

    # Plot Gaze Trajectory
    plt.plot(
        df["gaze_x"],
        df["gaze_y"], 
        label="Gaze Path",
        color="crimson",
        linewidth=1.5,
        linestyle="--",
        alpha=0.5,
    )

    # Highlight Start Point of the cursor
    plt.scatter(
        0,
        -360,
        color="green",
        s=100,
        zorder=5,
        label="Start Position",
    )

    # # Highlight End Point of the cursor
    # plt.scatter(
    #     df["cursor_x_cm"].iloc[-1],
    #     df["cursor_y_cm"].iloc[-1],
    #     color="red",
    #     s=100,
    #     zorder=5,
    #     label="End Position",
    # )

    # Plot Target Location
    plt.scatter(
        target_x,
        target_y,
        color="gold",
        edgecolors="black",
        s=200,
        marker="*",
        zorder=6,
        label="Target",
    )

    # 4. Customize plot appearance
    plt.title(f"Participant: {participant_id} - Block: {block_n} - Trial: {trial_n}", fontsize=14)
    plt.xlabel("X Position (pix)", fontsize=12)
    plt.ylabel("Y Position (pix)", fontsize=12)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axis("equal")  # Keeps the aspect ratio 1:1 so distances aren't distorted

    # 5. Display the plot
    plt.show()



### Visualize trial GUI

In [ ]:

# 2. Set up the state variables
participant_list = data['participant_id'].unique().tolist()
total_participants = len(participant_list)

# Track indices instead of hardcoded totals
current_participant = 0
current_block = 0
current_trial = 0

# 3. Create the UI components
output_area = widgets.Output()
prev_participant_button = widgets.Button(description="Prev participant", icon="arrow-left")
next_participant_button = widgets.Button(description="Next participant", icon="arrow-right")
prev_block_button = widgets.Button(description="Prev Block", icon="arrow-left")
next_block_button = widgets.Button(description="Next Block", icon="arrow-right")
prev_trial_button = widgets.Button(description="Prev Trial", icon="arrow-left")
next_trial_button = widgets.Button(description="Next Trial", icon="arrow-right")

# 4. Define button click behaviors

def on_prev_participant_clicked(b):
    global current_participant, current_block, current_trial
    if current_participant > 0:
        current_participant -= 1
        current_block = 0  # Reset to first block of new participant
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_next_participant_clicked(b):
    global current_participant, current_block, current_trial
    if current_participant < total_participants - 1:
        current_participant += 1
        current_block = 0  # Reset to first block of new participant
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_prev_block_clicked(b):
    global current_block, current_trial
    if current_block > 0:
        current_block -= 1
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_next_block_clicked(b):
    global current_block, current_trial
    # Bounds check uses dynamic data
    participant_id = participant_list[current_participant]
    total_blocks = data[data['participant_id'] == participant_id]['block'].nunique()
    
    if current_block < total_blocks - 1:
        current_block += 1
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_prev_trial_clicked(b):
    global current_trial
    if current_trial > 0:
        current_trial -= 1
        update_view()

def on_next_trial_clicked(b):
    global current_trial
    # Bounds check uses dynamic data
    participant_id = participant_list[current_participant]
    participant_data = data[data['participant_id'] == participant_id]
    block_list = participant_data['block'].unique().tolist()
    block_n = block_list[current_block]
    total_trials = participant_data[participant_data['block'] == block_n]['trial'].nunique()
    
    if current_trial < total_trials - 1:
        current_trial += 1
        update_view()

# 5. Define the update logic
def update_view():
    global current_block, current_trial
    
    # Extract current active IDs
    participant_id = participant_list[current_participant]
    participant_data = data[data['participant_id'] == participant_id]
    
    # Dynamically find available blocks
    block_list = participant_data['block'].unique().tolist()
    total_blocks = len(block_list)
    
    # Fallback safety check if indices get out of bounds during switch
    if current_block >= total_blocks:
        current_block = total_blocks - 1
        
    block_n = block_list[current_block]
    
    # Dynamically find available trials for this block
    # Note: Assumes your column name is 'trial_n'. Change if it's 'trial_id'.
    trial_list = participant_data[participant_data['block'] == block_n]['trial'].unique().tolist()
    total_trials = len(trial_list)
    
    if current_trial >= total_trials:
        current_trial = total_trials - 1
        
    trial_n = trial_list[current_trial]

    # Render the visualization
    with output_area:
        clear_output(wait=True)
        
        visualize_trial(data=data, participant_id=participant_id,block_n=block_n, trial_n=trial_n)
    
    # Disable buttons if at boundaries
    prev_participant_button.disabled = (current_participant == 0)
    next_participant_button.disabled = (current_participant == total_participants - 1)
    prev_block_button.disabled = (current_block == 0)
    next_block_button.disabled = (current_block == total_blocks - 1)
    prev_trial_button.disabled = (current_trial == 0)
    next_trial_button.disabled = (current_trial == total_trials - 1)

# 6. Bind callbacks and display the dashboard
prev_participant_button.on_click(on_prev_participant_clicked)
next_participant_button.on_click(on_next_participant_clicked)
prev_block_button.on_click(on_prev_block_clicked)
next_block_button.on_click(on_next_block_clicked)
prev_trial_button.on_click(on_prev_trial_clicked)
next_trial_button.on_click(on_next_trial_clicked)

button_layout = widgets.HBox([prev_participant_button, next_participant_button, prev_block_button, next_block_button, prev_trial_button, next_trial_button])
display(button_layout, output_area)

# Trigger the initial render
update_view()


Output()